In [1]:
# commons
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# pipelines
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import (
    ColumnTransformer, 
    make_column_selector,
    TransformedTargetRegressor
)

# proprocessing
from sklearn.model_selection import (
    train_test_split,
    cross_val_predict, 
    cross_validate,
    KFold,
    GridSearchCV
)

from sklearn.impute import SimpleImputer
from sklearn.feature_selection import (
    SelectPercentile, 
    chi2,
    RFECV,
    RFE
)
from sklearn.preprocessing import (
    FunctionTransformer,
    StandardScaler,
    MinMaxScaler,
    OneHotEncoder,
    OrdinalEncoder,
    PowerTransformer,
    QuantileTransformer,
    TargetEncoder
)
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from scipy.sparse import hstack


# models
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    AdaBoostRegressor,
    StackingRegressor,
    HistGradientBoostingRegressor
)
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import (
    LinearRegression,
    RidgeCV,
    LassoCV,
)

# metrics
from sklearn.metrics import (
    PredictionErrorDisplay,
    median_absolute_error, 
    mean_squared_error,
    mean_squared_log_error,
    r2_score
)
from sklearn.inspection import permutation_importance

# saving
from tempfile import mkdtemp
from shutil import rmtree
cachedir = mkdtemp()
from joblib import Memory

# utils
import time

# configs
sns.set_theme(style="white")

In [2]:
from data_transformers import drop_features, replace_none_with_nan, convert_columns_to_numeric


In [11]:
# Load data
raw_train_df = pd.read_parquet("../datasets/raw_train_oct_2024.snappy.parquet")
raw_test_df = pd.read_parquet("../datasets/raw_test_oct_2024.snappy.parquet")
raw_val_df = pd.read_parquet("../datasets/raw_val_oct_2024.snappy.parquet")

In [12]:
X_train, y_train = (
    raw_train_df.drop(columns=['days_to_sell']), raw_train_df['days_to_sell'])
X_test, y_test = (
    raw_test_df.drop(columns=['days_to_sell']), raw_test_df['days_to_sell'])  # Target variable
X_val, y_val = (
    raw_val_df.drop(columns=['days_to_sell']), raw_val_df['days_to_sell'])  # Target variable


In [13]:

# sample from the data
train_sample_size = 0.05
test_sample_size = 0.1

sampled_train_df = raw_train_df.sample(frac=train_sample_size, random_state=42)
X_train_sample, y_train_sample = (
    sampled_train_df.drop(columns=['days_to_sell']), sampled_train_df['days_to_sell'])

sampled_test_df = raw_test_df.sample(frac=test_sample_size, random_state=42)
X_test_sample, y_test_sample = (
    sampled_test_df.drop(columns=['days_to_sell']), sampled_test_df['days_to_sell'])


def print_data_summary(X_train, y_train, X_test, y_test, X_train_sample, y_train_sample, X_test_sample, y_test_sample):
    print("\n--- Data Summary ---")
    print(f"Train Set:      X: {X_train.shape}, y: {y_train.shape} (Total Samples: {len(X_train)})")
    print(f"Test Set:       X: {X_test.shape}, y: {y_test.shape} (Total Samples: {len(X_test)})")
    print(f"Sampled Train:  X: {X_train_sample.shape}, y: {y_train_sample.shape} (Samples: {len(X_train_sample)})")
    print(f"Sampled Test:   X: {X_test_sample.shape}, y: {y_test_sample.shape} (Samples: {len(X_test_sample)})")
    
# Call the function with your datasets
print_data_summary(X_train, y_train, X_test, y_test, X_train_sample, y_train_sample, X_test_sample, y_test_sample)



--- Data Summary ---
Train Set:      X: (112086, 42), y: (112086,) (Total Samples: 112086)
Test Set:       X: (31136, 42), y: (31136,) (Total Samples: 31136)
Sampled Train:  X: (5604, 42), y: (5604,) (Samples: 5604)
Sampled Test:   X: (3114, 42), y: (3114,) (Samples: 3114)


In [6]:
DEFAULT_COLS_DROP = ["stock_item_id", "last_date_seen", "first_date_seen", "derivative_id", "first_registration_date"]
DEFAULT_ZERO_NUM_FEATURES = ['battery_range_miles', 'battery_usable_capacity_kwh']
DEFAULT_NON_ZERO_NUM_FEATURES = ['first_retailer_asking_price',
 'last_retailer_asking_price',
 'reviews_per_100_advertised_stock_last_12_months',
 'seats',
 'doors',
 'co2_emission_gpkm',
 'top_speed_mph',
 'zero_to_sixty_mph_seconds',
 'engine_power_bhp',
 'fuel_economy_wltp_combined_mpg',
 'length_mm',
 'boot_space_seats_up_litres',
 'insurance_group',
 'plate',
 'odometer_reading_miles',
 'adjusted_retail_amount_gbp',
 'predicted_mileage',
 'number_of_images',
 'advert_quality']

In [7]:

# Updated Data Preparation Pipeline
data_preparation_pipeline = Pipeline([
    ('drop_columns', FunctionTransformer(drop_features)),  # Now imported
    ('replace_none', FunctionTransformer(replace_none_with_nan)),
    ('convert_2numeric', FunctionTransformer(convert_columns_to_numeric))
])


# Numerical Column transformer
numerical_transformer = ColumnTransformer(
    transformers=[
        ('zero_impute', SimpleImputer(
            strategy='constant', fill_value=0), DEFAULT_ZERO_NUM_FEATURES), # Zero imputation for specific columns
        ('imputation', SimpleImputer(strategy='mean'), DEFAULT_NON_ZERO_NUM_FEATURES) # Mean imputation for other numerical columns
    ]
)

# Numerical pipeline
numeric_pipeline = Pipeline([
    ('imputer', numerical_transformer),  # Handle missing values
    ('scaler', PowerTransformer(method="yeo-johnson"))  # Scale numeric features
])

cat_columns = ['can_home_deliver', 'segment', 'make', 'model', 'generation',
       'derivative', 'body_type', 'fuel_type', 'transmission_type',
       'drivetrain', 'colour', 'attention_grabber', 'manufacturer_approved',
       'price_indicator_rating', 'first_image_label', 'postcode_area']
n_unique_categories = X_train[cat_columns].nunique().sort_values(ascending=False)
high_cardinality_features = n_unique_categories[n_unique_categories > 255].index
low_cardinality_features = n_unique_categories[n_unique_categories <= 255].index


# Categorical transformer
mixed_encoded_preprocessor = ColumnTransformer(
    [
        (
            "high_cardinality",
            TargetEncoder(target_type="continuous"),
            high_cardinality_features,
        ),
        (
            "low_cardinality",
            OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
            low_cardinality_features,
        )
    ],
    verbose_feature_names_out=False,
)


columns_preprocessor = ColumnTransformer([
    ('numeric_features', numeric_pipeline, make_column_selector(dtype_include=np.number)),
    ('categorical_features', mixed_encoded_preprocessor, make_column_selector(dtype_include=["object", "bool"]))
])

# transformer
rf = RandomForestRegressor(random_state=42)
gbr = HistGradientBoostingRegressor(random_state=0)

min_features_to_select = 1  # Minimum number of features to consider
cv = KFold(5, shuffle=True, random_state=42)

rfecv_best = RFECV(
    estimator=rf,
    step=1,
    cv=cv,
    scoring="r2",
    min_features_to_select=min_features_to_select,
    n_jobs=-1,
)

quantile_transform = QuantileTransformer(n_quantiles=900, output_distribution="normal")

quantile_target_trans = TransformedTargetRegressor(
    regressor=gbr,
    transformer=quantile_transform,
)

# Define a cache directory
cache_dir = "./cache"
memory = Memory(cache_dir, verbose=0)

# Define the pipeline with caching
final_model_pipeline = Pipeline([
    ('data_preparation', data_preparation_pipeline),
    ('data_preprocessor', columns_preprocessor),
    ('rfe', rfecv_best),
    ('regressor_pipeline', quantile_target_trans)
], memory=memory)  # Enable caching


In [8]:
final_model_pipeline.fit(X_train_sample, y_train_sample)

Pipeline(memory=Memory(location=./cache/joblib),
         steps=[('data_preparation',
                 Pipeline(steps=[('drop_columns',
                                  FunctionTransformer(func=<function drop_features at 0x7a7d0bf20c20>)),
                                 ('replace_none',
                                  FunctionTransformer(func=<function replace_none_with_nan at 0x7a7dbdd151c0>)),
                                 ('convert_2numeric',
                                  FunctionTransformer(func=<function convert_columns_to_numeric at 0x7...
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x7a7d04320550>)])),
                ('rfe',
                 RFECV(cv=KFold(n_splits=5, random_state=42, shuffle=True),
                       estimator=RandomForestRegressor(random_state=42),
                       n_jobs=-1, scoring='r2')),
                ('regressor_pipeline',
                 TransformedTargetRegressor(regressor=HistGradientBoostingRegressor(random_state=0),
                                            transformer=QuantileTransformer(n_quantiles=900,
                                                                            output_distribution='normal')))])

## Best Parameter Grid search

In [10]:
scoring = {
    "R2": "r2"
}

# Define GridSearchCV
gs = GridSearchCV(
    final_model_pipeline,
    param_grid={
        "regressor_pipeline__regressor__min_samples_leaf": [10, 20, 50, 100, 200, 500]
    },
    scoring=scoring,
    refit="R2",
    n_jobs=-1,
    return_train_score=True,
)

# Fit the grid search (Cached transformations will be reused)
gs.fit(X_train_sample, y_train_sample)

# Get results
results = gs.cv_results_

print(f"Best parameters: {gs.best_params_}")
print(f"Best R² score: {gs.best_score_}")

Best parameters: {'regressor_pipeline__regressor__min_samples_leaf': 20}
Best R² score: 0.21379027775531156


In [14]:
# Print summary for Train, Validation, and Test datasets
print("Train Dataset Summary:")
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"Number of features: {X_train.shape[1]}")
print(f"Target variable: 'days_to_sell'")

print("\nValidation Dataset Summary:")
print(f"X_val shape: {X_val.shape}")
print(f"y_val shape: {y_val.shape}")
print(f"Number of features: {X_val.shape[1]}")
print(f"Target variable: 'days_to_sell'")

print("\nTest Dataset Summary:")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"Number of features: {X_test.shape[1]}")
print(f"Target variable: 'days_to_sell'")


Train Dataset Summary:
X_train shape: (112086, 42)
y_train shape: (112086,)
Number of features: 42
Target variable: 'days_to_sell'

Validation Dataset Summary:
X_val shape: (12454, 42)
y_val shape: (31136,)
Number of features: 42
Target variable: 'days_to_sell'

Test Dataset Summary:
X_test shape: (31136, 42)
y_test shape: (31136,)
Number of features: 42
Target variable: 'days_to_sell'


In [ ]:
# Best model parameters and pipeline setup
best_min_samples_leaf = 20  # Retrieved from GridSearchCV

best_model = HistGradientBoostingRegressor(
    random_state=0, min_samples_leaf=best_min_samples_leaf
)

final_model_pipeline.set_params(regressor_pipeline__regressor=best_model)
final_model_pipeline.fit(X_train, y_train)

# Get predictions for train, validation, and test data
y_train_pred = final_model_pipeline.predict(X_train)
y_val_pred = final_model_pipeline.predict(X_val)
y_test_pred = final_model_pipeline.predict(X_test)

In [ ]:
# Function to compute evaluation metrics
def compute_score(y_true, y_pred):
    return {
        "R2": f"{r2_score(y_true, y_pred):.3f}",
        "MedAE": f"{median_absolute_error(y_true, y_pred):.3f}",
    }

# Create the figure and subplots
f, ((ax0_0, ax0_1), (ax1_0, ax1_1)) = plt.subplots(2, 2, sharey="row", figsize=(6.5, 8))

# Plot actual vs predicted values for Train, Validation, and Test
PredictionErrorDisplay.from_predictions(
    y_train,
    y_train_pred,
    kind="actual_vs_predicted",
    ax=ax0_0,
    scatter_kwargs={"alpha": 0.5},
)
PredictionErrorDisplay.from_predictions(
    y_val,
    y_val_pred,
    kind="actual_vs_predicted",
    ax=ax0_1,
    scatter_kwargs={"alpha": 0.5},
)
PredictionErrorDisplay.from_predictions(
    y_test,
    y_test_pred,
    kind="actual_vs_predicted",
    ax=ax0_1,
    scatter_kwargs={"alpha": 0.5},
)

# Add R² and MedAE scores to the legend for each dataset
for ax, (y_true, y_pred, dataset) in zip(
    [ax0_0, ax0_1, ax0_1],
    [(y_train_sample, y_train_pred, "Train"), (y_val, y_val_pred, "Validation"), (y_test, y_test_pred, "Test")],
):
    for name, score in compute_score(y_true, y_pred).items():
        ax.plot([], [], " ", label=f"{name}={score}")
    ax.legend(loc="upper left")

ax0_0.set_title("Train Data \n Regression Performance")
ax0_1.set_title("Validation Data \n Regression Performance")

# Plot residuals vs predicted values for Train, Validation, and Test
PredictionErrorDisplay.from_predictions(
    y_train,
    y_train_pred,
    kind="residual_vs_predicted",
    ax=ax1_0,
    scatter_kwargs={"alpha": 0.5},
)
PredictionErrorDisplay.from_predictions(
    y_val,
    y_val_pred,
    kind="residual_vs_predicted",
    ax=ax1_1,
    scatter_kwargs={"alpha": 0.5},
)
PredictionErrorDisplay.from_predictions(
    y_test,
    y_test_pred,
    kind="residual_vs_predicted",
    ax=ax1_1,
    scatter_kwargs={"alpha": 0.5},
)

ax1_0.set_title("Train Data \n Residuals vs Predicted")
ax1_1.set_title("Validation Data \n Residuals vs Predicted")

# Set the overall title
f.suptitle("Auto Traders Cars Best Model Performance (Train, Validation, Test)", y=1.0)

# Adjust layout for better visualization
plt.tight_layout()
plt.show()

In [ ]:
# Print final scores for Train, Validation, and Test
print(f"Final Train R²: {r2_score(y_train_sample, y_train_pred):.3f}")
print(f"Final Validation R²: {r2_score(y_val_sample, y_val_pred):.3f}")
print(f"Final Test R²: {r2_score(y_test_sample, y_test_pred):.3f}")
print(f"Final Train MedAE: {median_absolute_error(y_train_sample, y_train_pred):.3f}")
print(f"Final Validation MedAE: {median_absolute_error(y_val_sample, y_val_pred):.3f}")
print(f"Final Test MedAE: {median_absolute_error(y_test_sample, y_test_pred):.3f}")
